# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

## Actividades

### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.

### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes

---
# Resolución del trabajo práctico

# Chatbot de preguntas frecuentes: **Asistente para postulantes a un Programa de Pasantías**

**Trabajo práctico – Procesamiento de Lenguaje Natural: chatbots por recuperación (TF-IDF vs. embeddings)**

| | |
|---|---|
| **Autor/a** | _Nicole Fernandez_ |
| **Fecha** | 12 octubre 2026 |

### Contenido
1. Definición de la aplicación y dataset de preguntas y respuestas (26 pares)
2. Chatbot 1: **TF-IDF + similitud del coseno**
3. Chatbot 2: **embeddings pre-entrenados de spaCy** (`es_core_news_md`) — elección y justificación
4. Ambos chatbots funcionando: set de prueba, relevamiento de aciertos y errores
5. Conclusiones: fallas, confusiones y comparación
6. Fuentes

## 1. Definición de la aplicación

### 1.1 ¿Qué problema resuelve?

El chatbot atiende a **estudiantes universitarios que quieren postularse al Programa de Pasantías de una empresa industrial** (empresa ficticia, sede en Buenos Aires y base operativa en Neuquén).

Durante cada convocatoria el equipo de Recursos Humanos (Recruiting) recibe cientos de consultas repetidas por mail y redes: *¿cuánto se cobra?*, *¿cuántas horas son?*, *¿puedo postularme si ya me recibí?*, *¿cómo sé en qué estado está mi postulación?*. Son preguntas con **respuesta fija y conocida**, ideales para un chatbot de **preguntas frecuentes (FAQ) por recuperación**: el bot no genera texto, sino que busca en una base cerrada la pregunta más parecida a la del usuario y devuelve su respuesta.

| Aspecto | Definición |
|---|---|
| **Usuario** | Estudiante universitario/terciario interesado en la pasantía |
| **Dominio** | Requisitos, postulación, proceso de selección, condiciones de la pasantía (duración, horas, asignación, beneficios), documentación y contacto |
| **Fuera de alcance** | Consultas personales sobre un caso puntual, temas ajenos (clima, compras, etc.). Para eso el bot responde con un mensaje de derivación |
| **Idioma** | Español rioplatense (voseo), con tildes o sin ellas, tono informal |
| **Marco normativo** | Las condiciones (duración, 20 h semanales, asignación estímulo, tutor, ART) respetan la **Ley 26.427 de Pasantías Educativas** de Argentina |

> **Decisión:** elegí este dominio porque es un caso de uso real de RRHH y permite construir preguntas de prueba realistas (con sinónimos y lenguaje coloquial, como escriben los postulantes).

In [ ]:
# ============ Configuración del entorno ============
# Se instalan spaCy y el modelo en español con vectores (≈ 40 MB).
!pip -q install "spacy>=3.8,<3.9"
!python -m spacy download es_core_news_md

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
print("Librerías cargadas")

### 1.2 Dataset de preguntas y respuestas

El dataset tiene **26 pares pregunta–respuesta** (el mínimo pedido es 20). Cada par tiene un `id` y una etiqueta de `intencion` que se usa solo para **evaluar** (saber si el bot devolvió la respuesta correcta); los chatbots no la usan para responder.

**Decisiones de diseño del dataset**
- Una pregunta "canónica" por intención, redactada como la escribiría RRHH en una página de FAQ.
- Respuestas concretas y autosuficientes (1–3 oraciones), con datos verificables cuando aplica (Ley 26.427: 2 a 12 meses, hasta 20 h semanales, asignación estímulo, cobertura de salud y ART, tutor y docente guía).
- Se incluyeron a propósito **intenciones cercanas entre sí** para poner a prueba a los bots: *duración de la pasantía* vs. *duración del proceso de selección*; *requisitos* vs. *promedio* vs. *graduados*; *etapas del proceso* vs. *evaluaciones online* vs. *entrevista*.

In [1]:
DATASET = [
    {"id": 1, "intencion": "requisitos",
     "pregunta": "¿Qué requisitos necesito para postularme a la pasantía?",
     "respuesta": "Tenés que ser estudiante regular de una carrera universitaria o terciaria, ser mayor de 18 años y tener aprobado al menos el 50 % de las materias. No hace falta experiencia laboral previa."},
    {"id": 2, "intencion": "como_postularse",
     "pregunta": "¿Cómo me postulo al programa de pasantías?",
     "respuesta": "La postulación es 100 % online: entrá al portal de empleos de la empresa, buscá 'Programa de Pasantías', completá el formulario y adjuntá tu CV en PDF."},
    {"id": 3, "intencion": "fechas_convocatoria",
     "pregunta": "¿Cuándo abre la convocatoria de pasantías?",
     "respuesta": "La convocatoria abre dos veces por año: en marzo (ingreso en junio) y en agosto (ingreso en noviembre). Las fechas exactas se publican en el portal de empleos y en nuestras redes."},
    {"id": 4, "intencion": "carreras",
     "pregunta": "¿Qué carreras buscan para las pasantías?",
     "respuesta": "Principalmente Ingenierías (Petróleo, Mecánica, Química, Industrial, Electrónica, Sistemas), Geología, Ciencias Económicas y Recursos Humanos. Cada búsqueda indica las carreras específicas."},
    {"id": 5, "intencion": "duracion_pasantia",
     "pregunta": "¿Cuánto dura la pasantía?",
     "respuesta": "La pasantía dura 6 meses y puede renovarse hasta completar 12. Por la Ley 26.427 la duración va de 2 a 12 meses, con una renovación posible de hasta 6 meses más."},
    {"id": 6, "intencion": "carga_horaria",
     "pregunta": "¿Cuántas horas por semana se trabaja?",
     "respuesta": "La carga es de 20 horas semanales (4 horas por día, de lunes a viernes), el máximo que permite la Ley de Pasantías. El horario se acuerda con tu tutor según tu cursada."},
    {"id": 7, "intencion": "remuneracion",
     "pregunta": "¿La pasantía es paga? ¿Cuánto se cobra?",
     "respuesta": "Sí. Recibís una asignación estímulo mensual (no remunerativa), calculada sobre el salario básico del convenio colectivo y proporcional a la carga horaria, como establece la Ley 26.427."},
    {"id": 8, "intencion": "modalidad",
     "pregunta": "¿La pasantía es presencial o remota?",
     "respuesta": "La modalidad es híbrida: 3 días presenciales en la oficina o planta y 2 días de trabajo remoto. Algunas posiciones de campo son 100 % presenciales."},
    {"id": 9, "intencion": "sedes",
     "pregunta": "¿En qué ciudades están las sedes?",
     "respuesta": "Tenemos posiciones en Buenos Aires (oficinas centrales) y en Neuquén (base operativa). Cada búsqueda indica la sede."},
    {"id": 10, "intencion": "etapas_seleccion",
     "pregunta": "¿Cuáles son las etapas del proceso de selección?",
     "respuesta": "El proceso tiene 4 etapas: 1) análisis de CV, 2) evaluaciones online, 3) entrevista con Recursos Humanos y 4) entrevista técnica con el área. Al final hacemos una propuesta formal."},
    {"id": 11, "intencion": "tiempo_respuesta",
     "pregunta": "¿Cuánto tarda el proceso de selección?",
     "respuesta": "En general el proceso completo lleva entre 4 y 6 semanas desde el cierre de la convocatoria. Te avisamos por mail en cada etapa."},
    {"id": 12, "intencion": "estado_postulacion",
     "pregunta": "¿Cómo puedo saber en qué estado está mi postulación?",
     "respuesta": "Podés ver el estado entrando al portal de empleos con tu usuario, en la sección 'Mis postulaciones'. También te notificamos por mail cada cambio."},
    {"id": 13, "intencion": "formato_cv",
     "pregunta": "¿Qué formato debe tener el CV?",
     "respuesta": "Recomendamos un CV de 1 o 2 páginas en PDF, con datos de contacto, carrera y avance (materias aprobadas y promedio), idiomas, conocimientos técnicos y proyectos. No hace falta incluir foto."},
    {"id": 14, "intencion": "ingles",
     "pregunta": "¿Es necesario saber inglés?",
     "respuesta": "Sí, pedimos inglés intermedio (lectura técnica y conversación básica), porque gran parte de la documentación y las reuniones globales son en inglés. Lo evaluamos durante el proceso."},
    {"id": 15, "intencion": "promedio",
     "pregunta": "¿Hay un promedio mínimo para postularse?",
     "respuesta": "Sugerimos un promedio de 6 o más, pero no es excluyente: también valoramos tu avance en la carrera, los proyectos y las actividades extracurriculares."},
    {"id": 16, "intencion": "preparar_entrevista",
     "pregunta": "¿Cómo me preparo para la entrevista?",
     "respuesta": "Repasá tu CV, investigá la empresa y la industria, prepará ejemplos concretos de proyectos o trabajo en equipo (método STAR) y pensá preguntas para hacernos. Si es virtual, conectate 5 minutos antes."},
    {"id": 17, "intencion": "evaluaciones_online",
     "pregunta": "¿Qué evaluaciones online hay que rendir?",
     "respuesta": "Son pruebas online de razonamiento lógico, numérico y verbal, más un cuestionario de preferencias de comportamiento. Duran unos 60 minutos y las hacés desde tu casa."},
    {"id": 18, "intencion": "beneficios",
     "pregunta": "¿Qué beneficios tiene la pasantía?",
     "respuesta": "Tenés obra social, cobertura de ART, días de licencia por examen, capacitaciones internas, comedor o reintegro de almuerzo y acceso a la plataforma de cursos online de la empresa."},
    {"id": 19, "intencion": "efectivizacion",
     "pregunta": "¿Existe la posibilidad de quedar efectivo al terminar la pasantía?",
     "respuesta": "Sí. Al finalizar evaluamos tu desempeño y, si hay vacantes, podés ingresar al Programa de Jóvenes Profesionales. Muchos de nuestros profesionales empezaron como pasantes."},
    {"id": 20, "intencion": "graduados",
     "pregunta": "¿Puedo postularme si ya me recibí?",
     "respuesta": "No. La Ley de Pasantías Educativas es para estudiantes, así que si ya te recibiste no podés ser pasante. Te invitamos a postularte al Programa de Jóvenes Profesionales."},
    {"id": 21, "intencion": "varias_busquedas",
     "pregunta": "¿Puedo postularme a más de una búsqueda a la vez?",
     "respuesta": "Sí, podés postularte hasta a 3 búsquedas de pasantía al mismo tiempo. Te recomendamos elegir las que mejor se ajusten a tu carrera e intereses."},
    {"id": 22, "intencion": "volver_a_postular",
     "pregunta": "Si no quedé seleccionado, ¿puedo volver a postularme?",
     "respuesta": "¡Claro! Podés volver a postularte en la próxima convocatoria. Te sugerimos actualizar tu CV con los nuevos avances en tu carrera."},
    {"id": 23, "intencion": "documentacion",
     "pregunta": "¿Qué documentación tengo que presentar si quedo seleccionado?",
     "respuesta": "Vas a necesitar DNI, certificado de alumno regular, certificado analítico, constancia de CUIL y el acuerdo de pasantía firmado por vos, tu universidad y la empresa. Si tu universidad no tiene convenio con nosotros, lo gestionamos."},
    {"id": 24, "intencion": "tutor",
     "pregunta": "¿Voy a tener un tutor durante la pasantía?",
     "respuesta": "Sí. Cada pasante tiene un tutor de la empresa que lo guía en el día a día y un docente guía de la universidad, como exige la Ley 26.427. Además hay reuniones de seguimiento con RRHH."},
    {"id": 25, "intencion": "contacto_rrhh",
     "pregunta": "¿Cómo me comunico con Recursos Humanos?",
     "respuesta": "Escribinos a pasantias@empresa-ejemplo.com o usá el chat del portal de empleos. Respondemos de lunes a viernes de 9 a 18 h."},
    {"id": 26, "intencion": "inclusion",
     "pregunta": "¿Pueden postularse personas con discapacidad?",
     "respuesta": "Sí, el programa es inclusivo. Si necesitás algún ajuste en el proceso de selección (formato de las pruebas, accesibilidad edilicia, intérprete), indicalo en el formulario y lo coordinamos."},
]

df_faq = pd.DataFrame(DATASET)
df_faq.to_csv("dataset_pasantias.csv", index=False, encoding="utf-8")
print(f"Pares pregunta-respuesta: {len(df_faq)}")
display(df_faq[["id", "intencion", "pregunta", "respuesta"]])

NameError: name 'pd' is not defined

## 2. Preprocesamiento común

Ambos chatbots usan **el mismo preprocesamiento** para que la comparación sea justa (la única diferencia es cómo se representa el texto):

| Paso | Decisión | Por qué |
|---|---|---|
| Minúsculas | Sí | "Pasantía" y "pasantía" deben ser la misma palabra. Además los vectores de spaCy distinguen mayúsculas. |
| Tokenización | Expresión regular `\w+` | Simple y predecible; separa palabras y descarta signos (`¿?¡!,`). |
| Tildes | TF-IDF: se **quitan**. Embeddings: se **conservan** | Los usuarios escriben "pasantia" o "pasantía": para TF-IDF deben coincidir. En embeddings se conservan porque el vocabulario de spaCy tiene ambas formas y la forma con tilde suele tener mejor vector. |
| Stopwords | Lista propia en español (artículos, preposiciones, pronombres, interrogativos, verbos auxiliares) | No aportan significado para elegir la respuesta y, en embeddings, "diluyen" el promedio. Se definió una lista propia (en vez de la de NLTK) para controlar qué se quita y que el notebook no dependa de descargas. |
| Lematización / stemming | No | Con TF-IDF ayudaría ("postulo"/"postularme"), pero quise ver el comportamiento "puro" de cada método y discutirlo en las conclusiones. |

In [ ]:
STOPWORDS = set('''
a al algo alguna algunas alguno algunos ante antes como con contra cual cuales cuando de del desde donde dos
el ella ellas ellos en entre era es esa esas ese eso esos esta estan estar este esto estos fue ha hay la las le les
lo los mas me mi mis mucho muy nada ni no nos o otra otro para pero poco por porque que quien se sea ser si sin
sobre su sus tambien te tengo tiene tu tus un una uno unos y ya yo vos voy va puedo puede
cuanto cuanta cuantos cuantas
'''.split())

def quitar_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

STOPWORDS = {quitar_tildes(w) for w in STOPWORDS}

def tokenizar(texto):
    """Minúsculas + tokens alfanuméricos (conserva tildes)."""
    return re.findall(r"\w+", texto.lower())

def es_stopword(token):
    return quitar_tildes(token) in STOPWORDS

def preprocesar_tfidf(texto):
    """Para TF-IDF: minúsculas, sin tildes, sin stopwords."""
    return " ".join(quitar_tildes(t) for t in tokenizar(texto) if not es_stopword(t))

for ejemplo in ["¿Cuánto dura la pasantía?", "que necesito para aplicar a una pasantia"]:
    print(f"{ejemplo!r:50} -> {preprocesar_tfidf(ejemplo)!r}")

## 2. Chatbot 1: TF-IDF + similitud del coseno

**Cómo funciona**
1. Cada entrada del dataset se convierte en un vector TF-IDF: cada dimensión es un término y su peso es alto si el término es frecuente en ese documento y raro en el resto (*term frequency × inverse document frequency*).
2. La pregunta del usuario se vectoriza con el **mismo** vocabulario.
3. Se calcula la **similitud del coseno** entre la pregunta y cada documento; se elige el de mayor puntaje.
4. Si el mejor puntaje no supera un **umbral**, el bot no arriesga: responde con un mensaje de derivación (*fallback*).

**Configuración de `TfidfVectorizer` y por qué**

| Parámetro | Valor | Justificación |
|---|---|---|
| `preprocessor` | `preprocesar_tfidf` | Minúsculas, sin tildes, sin stopwords (sección anterior). |
| `ngram_range` | `(1, 2)` | Los bigramas capturan expresiones como "proceso seleccion" u "obra social", que distinguen intenciones parecidas. |
| `sublinear_tf` | `True` | Usa 1 + log(tf): evita que una palabra repetida domine el vector. |
| Corpus indexado | **pregunta + respuesta** | Ver experimento de la sección 4.1: indexar solo la pregunta deja muy pocas palabras por documento y casi ninguna consulta reformulada coincide. La respuesta agrega vocabulario relacionado ("asignación", "salario", "remoto", "ART"). |
| Umbral | elegido en 4.2 | Se calibra con el set de prueba. |

In [ ]:
FALLBACK = ("Perdón, no tengo una respuesta para eso 🙁. Podés reformular la pregunta "
            "o escribir a pasantias@empresa-ejemplo.com y te responde el equipo de RRHH.")

def texto_indexado(item, campos):
    return " ".join(item[c] for c in campos)

class ChatbotTFIDF:
    def __init__(self, dataset, campos=("pregunta", "respuesta"), umbral=0.10):
        self.dataset = dataset
        self.umbral = umbral
        self.vectorizador = TfidfVectorizer(preprocessor=preprocesar_tfidf,
                                            ngram_range=(1, 2), sublinear_tf=True)
        self.matriz = self.vectorizador.fit_transform([texto_indexado(d, campos) for d in dataset])

    def puntajes(self, pregunta):
        return cosine_similarity(self.vectorizador.transform([pregunta]), self.matriz)[0]

    def predecir(self, pregunta):
        """Devuelve (id, puntaje); id = 0 si no supera el umbral."""
        s = self.puntajes(pregunta)
        i = int(np.argmax(s))
        return (self.dataset[i]["id"] if s[i] >= self.umbral else 0), float(s[i])

    def responder(self, pregunta):
        id_, _ = self.predecir(pregunta)
        return FALLBACK if id_ == 0 else next(d["respuesta"] for d in self.dataset if d["id"] == id_)

bot_tfidf = ChatbotTFIDF(DATASET)
print("Tamaño del vocabulario (unigramas + bigramas):", len(bot_tfidf.vectorizador.vocabulary_))
print("Matriz documentos x términos:", bot_tfidf.matriz.shape)

In [ ]:
# ¿Qué términos pesan más en un documento? (ej.: id 7, remuneración)
terminos = np.array(bot_tfidf.vectorizador.get_feature_names_out())
fila = bot_tfidf.matriz[6].toarray()[0]
top = fila.argsort()[::-1][:8]
print("Términos con más peso en el documento 7 (remuneración):")
for t, w in zip(terminos[top], fila[top]):
    print(f"  {t:28} {w:.3f}")

## 3. Chatbot 2: embeddings pre-entrenados

### 3.1 ¿Qué embedding elegí? → **spaCy `es_core_news_md`** (vectores fastText de 300 dimensiones)

| Opción evaluada | Idioma | ¿Sirve para este caso? |
|---|---|---|
| **spaCy `es_core_news_md`** ✅ | **Español** | Trae vectores estáticos de 300 dim. entrenados con fastText (CBOW) sobre **OSCAR Common Crawl + Wikipedia en español**. 500.000 palabras mapeadas a 20.000 vectores únicos. Pesa ~40 MB y se instala en Colab con un comando. |
| spaCy `es_core_news_lg` | Español | Mismos datos pero 500.000 vectores únicos (~550 MB). Mejor cobertura, pero mucho más pesado para un TP; queda como mejora posible. |
| spaCy `es_core_news_sm` | Español | **No** trae vectores de palabras (solo usa vectores de contexto del pipeline), por eso se descarta. |
| gensim (`gensim-data`) | Inglés | Los modelos pre-entrenados disponibles por el downloader (`word2vec-google-news-300`, `glove-wiki-gigaword-*`, `fasttext-wiki-news-subwords-300`) son en **inglés**: no sirven para consultas en español. |
| NLTK | Inglés | NLTK no ofrece embeddings pre-entrenados en español (solo una muestra de word2vec en inglés). |

**Justificación:** el chatbot recibe preguntas en español rioplatense, así que el requisito excluyente era un embedding **en español**. Entre las tres librerías pedidas, la única que ofrece uno listo para usar es spaCy. Elegí la versión `md` por el equilibrio tamaño/cobertura: incluye palabras coloquiales y sin tilde que usan los postulantes ("pasantia", "sueldo", "curriculum", "cv").

### 3.2 Cómo se arma el vector de una oración

Un embedding de palabras da un vector por palabra; para comparar oraciones hay que combinarlos. Probé **dos estrategias** (ambas con similitud del coseno):

- **A. Promedio ponderado por IDF** (enfoque principal): vector de la oración = promedio de los vectores de sus palabras (sin stopwords), ponderado por el IDF de cada palabra. El IDF se calcula sobre el mismo corpus que TF-IDF. **¿Por qué ponderar?** Con un promedio simple, palabras comunes del dominio como "pasantía" o "postular" pesan igual que las que realmente definen la intención ("sueldo", "inglés"), y todas las oraciones terminan pareciéndose.
- **B. Alineación palabra a palabra** (variante): para cada palabra de la consulta se busca la palabra más parecida del documento (coseno entre vectores) y se promedian esos máximos, ponderados por IDF. Es una versión simplificada de la idea de BERTScore/soft-cosine con vectores estáticos. Evita que el promedio "mezcle" significados en un solo vector.

Las palabras **sin vector** en el modelo se ignoran (si no queda ninguna, el puntaje es 0 → fallback).

In [ ]:
import spacy
nlp = spacy.load("es_core_news_md")
print("Modelo:", nlp.meta["name"], nlp.meta["version"])
print("Tabla de vectores (vectores únicos, dimensión):", nlp.vocab.vectors.shape)
print("Palabras con vector:", nlp.vocab.vectors.n_keys)

def vector(palabra):
    return nlp.vocab[palabra].vector

def coseno(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return float(u @ v / (nu * nv)) if nu and nv else 0.0

# ¿Capturan sinónimos? (lo que TF-IDF NO puede hacer)
pares = [("sueldo", "salario"), ("sueldo", "asignación"), ("pagan", "cobra"), ("inscribo", "postulo"),
         ("curriculum", "cv"), ("graduado", "recibí"), ("home", "remoto"), ("sueldo", "empanadas")]
for a, b in pares:
    print(f"  coseno({a!r:13}, {b!r:13}) = {coseno(vector(a), vector(b)):.2f}")

In [ ]:
class ChatbotEmbeddings:
    def __init__(self, dataset, modo="promedio", campos=("pregunta", "respuesta"), umbral=0.5):
        assert modo in ("promedio", "alineacion")
        self.dataset, self.modo, self.umbral = dataset, modo, umbral
        textos = [texto_indexado(d, campos) for d in dataset]
        # IDF (unigramas) calculado sobre el mismo corpus, para ponderar palabras
        tfidf = TfidfVectorizer(preprocessor=preprocesar_tfidf).fit(textos)
        self.idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
        self.idf_max = max(self.idf.values())  # palabras no vistas: máxima rareza
        self.docs = [self._palabras_y_vectores(t) for t in textos]
        self.doc_vecs = np.array([self._vector_oracion(p, V) for p, V in self.docs])

    def _palabras_y_vectores(self, texto):
        palabras = [t for t in tokenizar(texto) if not es_stopword(t) and nlp.vocab.has_vector(t)]
        V = np.array([vector(p) / np.linalg.norm(vector(p)) for p in palabras]) if palabras else np.zeros((0, 300))
        return palabras, V

    def _pesos(self, palabras):
        return np.array([self.idf.get(quitar_tildes(p), self.idf_max) for p in palabras])

    def _vector_oracion(self, palabras, V):
        if not palabras:
            return np.zeros(300)
        return np.average(V, axis=0, weights=self._pesos(palabras))

    def puntajes(self, pregunta):
        palabras, V = self._palabras_y_vectores(pregunta)
        if not palabras:
            return np.zeros(len(self.dataset))
        if self.modo == "promedio":
            q = self._vector_oracion(palabras, V)
            return np.array([coseno(q, d) for d in self.doc_vecs])
        w = self._pesos(palabras)            # modo "alineacion"
        return np.array([float(((V @ Vd.T).max(axis=1) * w).sum() / w.sum()) for _, Vd in self.docs])

    def predecir(self, pregunta):
        s = self.puntajes(pregunta)
        i = int(np.argmax(s))
        return (self.dataset[i]["id"] if s[i] >= self.umbral else 0), float(s[i])

    def responder(self, pregunta):
        id_, _ = self.predecir(pregunta)
        return FALLBACK if id_ == 0 else next(d["respuesta"] for d in self.dataset if d["id"] == id_)

bot_emb = ChatbotEmbeddings(DATASET, modo="promedio")
bot_emb_al = ChatbotEmbeddings(DATASET, modo="alineacion")
sin_vector = sorted({t for d in DATASET for t in tokenizar(d["pregunta"] + " " + d["respuesta"])
                     if not es_stopword(t) and not nlp.vocab.has_vector(t)})
print("Palabras del dataset sin vector en el modelo:", sin_vector)

## 4. Ambos chatbots funcionando

### Set de prueba

Se armaron **37 preguntas de prueba**, **distintas** a las del dataset (si se probara con las mismas preguntas, ambos bots acertarían el 100 % y no se aprendería nada):

- **32 preguntas dentro del dominio** (al menos una por intención), con distintas dificultades: sinónimos ("sueldo" en lugar de "asignación"), lenguaje coloquial ("cuánto pagan?", "tips para la entrevista"), sin tildes ("pasantia"), anglicismos ("home office") y **preguntas trampa** que comparten palabras con otra intención ("cuál es la *duración* del proceso de selección").
- **5 preguntas fuera de dominio** (clima, compras, receta, acciones, "hola"): lo correcto es que el bot **no** responda y derive (`id_esperado = 0`).

**Criterio de acierto:** dentro del dominio, el bot acierta si devuelve la respuesta de la intención esperada; fuera de dominio, si responde con el fallback.

In [ ]:
PRUEBAS = [
    ("que necesito para aplicar a una pasantia", 1),
    ("Cuáles son los requisitos?", 1),
    ("dónde me inscribo para la pasantía", 2),
    ("cuando abren las inscripciones", 3),
    ("buscan estudiantes de ingeniería química?", 4),
    ("cuántos meses dura", 5),
    ("cuál es el horario de trabajo", 6),
    ("cuántas horas tengo que trabajar por semana", 6),
    ("cuánto pagan?", 7),
    ("cuál es el sueldo de un pasante", 7),
    ("se puede hacer home office?", 8),
    ("la pasantía es en la oficina o desde casa", 8),
    ("trabajan en Neuquén?", 9),
    ("cómo es el proceso de selección", 10),
    ("en cuánto tiempo me responden", 11),
    ("cuál es la duración del proceso de selección", 11),
    ("quiero saber cómo va mi postulación", 12),
    ("el curriculum tiene que llevar foto?", 13),
    ("piden nivel de inglés?", 14),
    ("necesito tener buen promedio?", 15),
    ("tips para la entrevista", 16),
    ("qué tipo de tests psicotécnicos toman", 17),
    ("tienen obra social?", 18),
    ("qué beneficios dan", 18),
    ("después de la pasantía me pueden contratar?", 19),
    ("soy graduado, puedo aplicar?", 20),
    ("puedo anotarme en varias búsquedas", 21),
    ("me rechazaron, puedo intentar de nuevo?", 22),
    ("qué papeles tengo que llevar", 23),
    ("quién me va a acompañar en la pasantía", 24),
    ("cuál es el mail de contacto", 25),
    ("tengo una discapacidad, puedo postularme?", 26),
    ("cuál es el precio de las acciones de la empresa", 0),
    ("me recomendás una receta de empanadas", 0),
    ("qué tiempo va a hacer mañana", 0),
    ("quiero comprar una notebook", 0),
    ("hola", 0),
]

df_pruebas = pd.DataFrame(PRUEBAS, columns=["pregunta_prueba", "id_esperado"])
intencion_por_id = {d["id"]: d["intencion"] for d in DATASET} | {0: "(fuera de dominio)"}
df_pruebas["intencion_esperada"] = df_pruebas["id_esperado"].map(intencion_por_id)
print(f"Preguntas de prueba: {len(df_pruebas)}  (dentro del dominio: {(df_pruebas.id_esperado > 0).sum()}, fuera: {(df_pruebas.id_esperado == 0).sum()})")
display(df_pruebas)

### 4.1 Decisión de diseño: ¿indexar solo la pregunta o pregunta + respuesta?

Antes de fijar la configuración comparé, **sin umbral** (siempre devolviendo la mejor coincidencia), qué pasa si cada documento es solo la pregunta del FAQ o la pregunta más su respuesta. Se mide la exactitud sobre las 32 preguntas dentro del dominio.

In [ ]:
dentro = [(p, e) for p, e in PRUEBAS if e > 0]

def exactitud_top1(bot):
    return np.mean([bot.dataset[int(np.argmax(bot.puntajes(p)))]["id"] == e for p, e in dentro])

filas = []
for nombre, campos in [("solo pregunta", ("pregunta",)), ("pregunta + respuesta", ("pregunta", "respuesta"))]:
    filas.append({
        "corpus indexado": nombre,
        "TF-IDF": exactitud_top1(ChatbotTFIDF(DATASET, campos=campos)),
        "Embeddings A (promedio IDF)": exactitud_top1(ChatbotEmbeddings(DATASET, "promedio", campos=campos)),
        "Embeddings B (alineación)": exactitud_top1(ChatbotEmbeddings(DATASET, "alineacion", campos=campos)),
    })
df_corpus = pd.DataFrame(filas).set_index("corpus indexado")
display(df_corpus.style.format("{:.0%}"))

Indexar **pregunta + respuesta** mejora a los tres métodos, así que es la configuración que se usa en el resto del notebook.

### 4.2 Calibración del umbral (fallback)

Sin umbral el bot **siempre** responde algo, incluso a "qué tiempo va a hacer mañana". El umbral define cuándo prefiere derivar. Se recorre un rango de umbrales y se mide la exactitud sobre las **37** preguntas (dentro + fuera de dominio).

In [ ]:
bots = {"TF-IDF": bot_tfidf, "Embeddings A (promedio IDF)": bot_emb, "Embeddings B (alineación)": bot_emb_al}

# Puntaje máximo y predicción (sin umbral) de cada bot para cada prueba
crudo = {}
for nombre, bot in bots.items():
    filas = []
    for p, e in PRUEBAS:
        s = bot.puntajes(p); i = int(np.argmax(s))
        filas.append((DATASET[i]["id"], float(s[i]), e))
    crudo[nombre] = filas

def exactitud_con_umbral(filas, u):
    return np.mean([((pid if sc >= u else 0) == e) for pid, sc, e in filas])

umbrales = np.round(np.arange(0.0, 0.951, 0.01), 2)
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.2))
mejor_umbral = {}
for nombre, filas in crudo.items():
    acc = [exactitud_con_umbral(filas, u) for u in umbrales]
    mejor_umbral[nombre] = float(umbrales[int(np.argmax(acc))])
    ejes[0].plot(umbrales, acc, label=nombre)
ejes[0].set(title="Exactitud total (37 preguntas) según el umbral", xlabel="umbral", ylabel="exactitud")
ejes[0].legend(); ejes[0].grid(alpha=.3)

# Distribución de puntajes: aciertos, errores y fuera de dominio
for k, (nombre, filas) in enumerate(crudo.items()):
    for etiqueta, color, filtro in [("acierto", "tab:green", lambda pid, e: e > 0 and pid == e),
                                    ("error", "tab:red", lambda pid, e: e > 0 and pid != e),
                                    ("fuera de dominio", "tab:gray", lambda pid, e: e == 0)]:
        xs = [sc for pid, sc, e in filas if filtro(pid, e)]
        ejes[1].scatter(xs, [k + np.random.default_rng(k).uniform(-.15, .15) for _ in xs],
                        c=color, s=28, alpha=.8, label=etiqueta if k == 0 else None)
    ejes[1].axvline(mejor_umbral[nombre], ymin=(k + .1) / 3, ymax=(k + .9) / 3, color="k", ls="--", lw=1)
ejes[1].set_yticks(range(3), list(crudo)); ejes[1].set_xlabel("similitud del coseno máxima")
ejes[1].set_title("Puntaje de la mejor coincidencia (línea = umbral elegido)"); ejes[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

for nombre, u in mejor_umbral.items():
    bots[nombre].umbral = u
    print(f"Umbral elegido para {nombre:28}: {u:.2f}")

> **Nota metodológica:** el umbral se eligió con el mismo set de prueba con el que se evalúa, porque el dataset es chico. Eso hace que los resultados sean algo **optimistas**; con más datos lo correcto sería calibrar en un set de validación separado.

### 4.3 Resultados pregunta por pregunta (✅ correcta / ❌ incorrecta)

In [ ]:
def etiqueta(id_):
    return f"{id_} · {intencion_por_id[id_]}"

resultados = df_pruebas[["pregunta_prueba", "intencion_esperada", "id_esperado"]].copy()
for nombre, bot in bots.items():
    preds = [bot.predecir(p) for p in resultados["pregunta_prueba"]]
    corto = {"TF-IDF": "tfidf", "Embeddings A (promedio IDF)": "embA", "Embeddings B (alineación)": "embB"}[nombre]
    resultados[f"{corto}_pred"] = [etiqueta(i) for i, _ in preds]
    resultados[f"{corto}_score"] = [round(s, 2) for _, s in preds]
    resultados[f"{corto}_ok"] = [i == e for (i, _), e in zip(preds, resultados["id_esperado"])]

resultados.to_csv("resultados_pruebas.csv", index=False, encoding="utf-8")

vista = resultados[["pregunta_prueba", "intencion_esperada"]].copy()
for c in ["tfidf", "embA", "embB"]:
    vista[c] = [("✅ " if ok else "❌ ") + (pred.split(" · ")[1] if not ok else "") + f" ({sc:.2f})"
                for ok, pred, sc in zip(resultados[f"{c}_ok"], resultados[f"{c}_pred"], resultados[f"{c}_score"])]
vista.columns = ["Pregunta de prueba", "Intención esperada", "TF-IDF", "Emb. A (promedio)", "Emb. B (alineación)"]
print("Entre paréntesis: similitud máxima. En los errores se muestra la intención que devolvió el bot.\n")
display(vista)

In [ ]:
def resumen(col):
    ok = resultados[col]
    din = resultados["id_esperado"] > 0
    pred = resultados[col.replace("_ok", "_pred")]
    sin_responder = pred.str.startswith("0 ")
    return {"Aciertos totales (37)": f"{ok.sum()} / 37  ({ok.mean():.0%})",
            "Dentro del dominio (32)": f"{ok[din].sum()} / 32  ({ok[din].mean():.0%})",
            "Fuera de dominio bien derivadas (5)": f"{ok[~din].sum()} / 5",
            "Respuestas incorrectas dadas con seguridad": int(((~ok) & din & ~sin_responder).sum() + ((~ok) & ~din).sum()),
            "Preguntas válidas sin responder (fallback de más)": int(((~ok) & din & sin_responder).sum())}

df_resumen = pd.DataFrame({"TF-IDF": resumen("tfidf_ok"),
                           "Embeddings A (promedio IDF)": resumen("embA_ok"),
                           "Embeddings B (alineación)": resumen("embB_ok")})
display(df_resumen)

### 4.4 Conversaciones de ejemplo

Así responde cada bot a algunas preguntas del set de prueba (texto completo de la respuesta):

In [ ]:
ejemplos = ["cuánto pagan?", "se puede hacer home office?", "soy graduado, puedo aplicar?",
            "cuál es la duración del proceso de selección", "qué tiempo va a hacer mañana"]
for p in ejemplos:
    print("🧑 Usuario:", p)
    for nombre, bot in [("TF-IDF", bot_tfidf), ("Emb. A", bot_emb), ("Emb. B", bot_emb_al)]:
        id_, sc = bot.predecir(p)
        print(f"🤖 {nombre:7}[{etiqueta(id_)} | {sc:.2f}] {bot.responder(p)}")
    print("-" * 110)

### 4.5 Chat interactivo

Descomentá la última línea para conversar con el bot que quieras (escribí `salir` para terminar). Está comentada para que "Ejecutar todo" no quede esperando una entrada.

In [ ]:
def chatear(bot, nombre="PasantIA"):
    print(f"{nombre}: ¡Hola! Soy el asistente del Programa de Pasantías. Preguntame lo que necesites (o 'salir').")
    while True:
        pregunta = input("Vos: ").strip()
        if pregunta.lower() in {"salir", "chau", "exit"}:
            print(f"{nombre}: ¡Éxitos con tu postulación! 👋"); break
        print(f"{nombre}: {bot.responder(pregunta)}")

# chatear(bot_emb_al)   # ← o bot_tfidf / bot_emb

### 4.6 Análisis de errores: ¿qué preguntas confunde cada bot?

In [ ]:
for c, nombre in [("tfidf", "TF-IDF"), ("embA", "Embeddings A (promedio IDF)"), ("embB", "Embeddings B (alineación)")]:
    err = resultados[~resultados[f"{c}_ok"]]
    print(f"\n=== {nombre}: {len(err)} errores ===")
    for _, r in err.iterrows():
        print(f"  • {r.pregunta_prueba!r:52} esperado: {r.intencion_esperada:22} → devolvió: {r[c + '_pred']} ({r[c + '_score']:.2f})")

# Preguntas que fallan TODOS los bots y preguntas en las que solo uno acierta
todos = resultados[~(resultados.tfidf_ok | resultados.embA_ok | resultados.embB_ok)]
print("\nFallan los tres bots:", list(todos.pregunta_prueba))
solo_tfidf = resultados[resultados.tfidf_ok & ~resultados.embA_ok & ~resultados.embB_ok]
solo_emb = resultados[~resultados.tfidf_ok & (resultados.embA_ok | resultados.embB_ok)]
print("Solo acierta TF-IDF:", list(solo_tfidf.pregunta_prueba))
print("Aciertan los embeddings y TF-IDF no:", list(solo_emb.pregunta_prueba))

In [ ]:
# ¿Por qué TF-IDF da 0 en tantas preguntas? → ninguna palabra de la consulta existe en su vocabulario
vocab = bot_tfidf.vectorizador.vocabulary_
for p in ["cuánto pagan?", "cuál es el sueldo de un pasante", "se puede hacer home office?", "soy graduado, puedo aplicar?"]:
    toks = preprocesar_tfidf(p).split()
    print(f"{p!r:40} tokens: {toks}  →  en el vocabulario: {[t for t in toks if t in vocab]}")

## 5. Conclusiones

### 5.1 Resumen de resultados

| | TF-IDF | Embeddings A (promedio IDF) | Embeddings B (alineación) |
|---|---|---|---|
| Aciertos totales (37 preguntas) | 20 (54 %) | 18 (49 %) | **23 (62 %)** |
| Dentro del dominio (32) | 17 (53 %) | 17 (53 %) | **18 (56 %)** |
| Fuera de dominio derivadas (5) | 3 | 1 | **5** |
| Respuestas **incorrectas** dadas con seguridad | 10 | 19 | **7** |
| Preguntas válidas que dejó sin responder | 7 | **0** | 7 |

Ningún bot supera el 62 %: con **una sola pregunta de referencia por intención**, reconocer reformulaciones libres es difícil para cualquiera de los dos enfoques. Lo interesante es *cómo* falla cada uno.

### 5.2 ¿En qué respuestas falla cada bot? ¿Qué preguntas confunde?

**TF-IDF – falla por falta de coincidencia léxica (vocabulario).**
- Solo "entiende" palabras que aparecen literalmente en el dataset. En **7 preguntas el puntaje fue 0** porque ninguna palabra de la consulta estaba en su vocabulario: "cuánto **pagan**?", "se puede hacer **home office**?", "soy **graduado**, puedo **aplicar**?", "me **rechazaron**…", "qué **papeles**…", "**tests psicotécnicos**", "cuando **abren** las **inscripciones**". En el dataset dice "cobra", "remota", "me recibí", "documentación", "evaluaciones". Al no tener lematización, tampoco une "postulo" con "postularme".
- Cuando sí coincide alguna palabra, se deja llevar por **palabras compartidas**: "cómo es el *proceso de selección*" → *tiempo de respuesta* (la FAQ 11 dice "¿Cuánto tarda el proceso de selección?"); "tips para la *entrevista*" → *etapas del proceso* (la respuesta de etapas menciona dos entrevistas); "quién me va a acompañar en la *pasantía*" → *duración de la pasantía*.
- Su ventaja: es **literal y predecible**. Acertó la pregunta trampa "cuál es la *duración del proceso de selección*" (donde los dos bots de embeddings fallaron) y "necesito tener buen *promedio*?", porque la palabra exacta estaba en el documento correcto. Y casi no "inventa": ante temas ajenos suele dar 0 y derivar.

**Embeddings – resuelven sinónimos, pero mezclan significados.**
- Ganan justamente donde TF-IDF se queda en 0: "cuánto **pagan**?" y "cuál es el **sueldo**…" → *remuneración*; "**tests psicotécnicos**" → *evaluaciones online*; "**tips** para la entrevista"; "cuando **abren** las **inscripciones**" → *fechas de convocatoria*. Los vectores saben que *sueldo ≈ salario* (coseno 0,84) aunque esa palabra nunca aparezca en el dataset. En total, **11 preguntas** que TF-IDF falla (9 del dominio y 2 fuera de dominio) las acierta al menos un bot de embeddings.
- **Variante A (promedio)** es la más débil: al promediar todas las palabras de una pregunta + respuesta de ~40 palabras, los vectores de los documentos se parecen mucho entre sí (todos hablan de pasantía, postulación, empresa). Resultado: **puntajes altos para todo** (0,4–0,8), lo que hace imposible fijar un umbral útil: o responde cualquier cosa, o no responde nada. Por eso respondió con seguridad "volver a postularte" a *¿qué tiempo va a hacer mañana?* y *beneficios* a *¿precio de las acciones?* (19 respuestas incorrectas en total). Confunde intenciones del mismo "campo semántico": *beneficios ↔ efectivización ↔ inglés*, *modalidad ↔ tutor*, *requisitos ↔ formato de CV / preparar entrevista*.
- **Variante B (alineación palabra a palabra)** es la mejor en total: compara cada palabra de la consulta con su mejor pareja del documento, así una palabra clave ("sueldo"→"salario", "Neuquén", "discapacidad") pesa por sí sola. Separó bien lo fuera de dominio (**5/5**). Su costo: el umbral (0,63) que filtra lo ajeno también **descarta 7 preguntas válidas** que quedaron con puntaje entre 0,42 y 0,62 ("puedo anotarme en varias búsquedas", "me rechazaron…", "qué papeles tengo que llevar").

**Preguntas que confunden los tres bots**

| Pregunta | Esperada | Problema |
|---|---|---|
| "dónde me inscribo para la pasantía" | cómo postularse | "inscribo" no está en el dataset; las intenciones *requisitos*, *cómo postularse* y *documentación* comparten vocabulario. |
| "se puede hacer home office?" | modalidad | Anglicismo: "home" y "office" no se relacionan con "remota/híbrida" ni por palabra ni por vector. |
| "en cuánto tiempo me responden" | tiempo de respuesta | "tiempo" coincide con "al mismo **tiempo**" de la FAQ de varias búsquedas: **ambigüedad léxica** (tiempo = duración / momento). |
| "soy graduado, puedo aplicar?" | graduados | La FAQ dice "si ya me recibí": la relación *graduado ↔ recibirse* es de conocimiento del mundo, no de similitud de palabras. |
| "qué papeles tengo que llevar" | documentación | "papeles" en el sentido coloquial de "documentos" no está bien representado. |
| "cuál es el mail de contacto" | contacto RRHH | "mail" arrastra hacia las FAQ que dicen "te notificamos / te avisamos por **mail**" (*estado de postulación*, *tiempo de respuesta*), y la respuesta correcta tiene la dirección de correo pero no la palabra "mail". |

### 5.3 Comparación y aprendizajes

1. **TF-IDF mide coincidencia de palabras; los embeddings miden cercanía de significado.** TF-IDF falla por *omisión* (no reconoce sinónimos y no responde) y los embeddings fallan por *exceso* (encuentran parecido en todo). Para un chatbot de atención esto importa: **una respuesta equivocada dicha con seguridad es peor que un "no sé, escribí a RRHH"**. Con ese criterio, el promedio de embeddings (A) es el peor, aunque acierte las mismas preguntas dentro del dominio que TF-IDF.
2. **La forma de combinar los vectores importa tanto como el embedding.** Con el mismo modelo de spaCy, pasar de promediar (A) a alinear palabra a palabra (B) subió de 18 a 23 aciertos y de 1 a 5 rechazos correctos.
3. **Indexar la respuesta ayuda y molesta.** Mejoró a los tres métodos (sección 4.1), pero también introdujo ruido: la FAQ de *inclusión* contiene "proceso de selección" y por eso la variante B la eligió para "duración del proceso de selección"; la respuesta de *etapas* menciona "entrevista" y atrae a "tips para la entrevista" en TF-IDF.
4. **El umbral es una decisión de negocio, no solo técnica.** En TF-IDF los puntajes son bajos (0,1–0,4) y el 0 indica claramente "no sé"; en embeddings todo puntúa entre 0,4 y 1, así que el umbral queda en una zona donde se mezclan aciertos, errores y preguntas ajenas (gráfico de 4.2).
5. **Límites del experimento:** 26 intenciones, 37 preguntas de prueba y un umbral calibrado con el mismo set de prueba, así que los porcentajes son orientativos y algo optimistas.

### 5.4 Mejoras propuestas
- **Más ejemplos por intención** (3–5 formas de preguntar lo mismo, sacadas de consultas reales de postulantes): es la mejora con mayor impacto para ambos bots.
- **Lematización** con spaCy (`token.lemma_`) para TF-IDF ("postulo", "postularme", "postulación").
- **Sistema híbrido:** combinar el puntaje de TF-IDF (preciso cuando hay coincidencia exacta) con el de embeddings (cubre sinónimos), o usar TF-IDF como primer filtro.
- **Glosario de sinónimos del dominio** (home office → remoto, papeles → documentación, graduado → recibido).
- **Embeddings de oraciones** (p. ej. `sentence-transformers` multilingüe) o el modelo `es_core_news_lg`, que representan mejor el significado de frases completas.
- Mostrar al usuario **las 2–3 preguntas más parecidas** cuando el puntaje es dudoso ("¿Quisiste decir…?") en lugar de elegir una sola.

## 6. Fuentes

**Herramientas y modelos**
- scikit-learn — `TfidfVectorizer`: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- scikit-learn — `cosine_similarity`: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- spaCy — modelo `es_core_news_md` (vectores: *Explosion fastText Vectors, cbow, OSCAR Common Crawl + Wikipedia*): https://spacy.io/models/es#es_core_news_md
- spaCy — Vectores y similitud: https://spacy.io/usage/linguistic-features#vectors-similarity
- gensim — modelos pre-entrenados disponibles (`gensim-data`): https://github.com/piskvorky/gensim-data
- NLTK Data: https://www.nltk.org/nltk_data/

**Bibliografía**
- Salton, G. & Buckley, C. (1988). *Term-weighting approaches in automatic text retrieval*. Information Processing & Management, 24(5), 513–523.
- Mikolov, T., Chen, K., Corrado, G. & Dean, J. (2013). *Efficient Estimation of Word Representations in Vector Space*. arXiv:1301.3781.
- Bojanowski, P., Grave, E., Joulin, A. & Mikolov, T. (2017). *Enriching Word Vectors with Subword Information*. TACL, 5, 135–146.
- Ortiz Suárez, P. J., Sagot, B. & Romary, L. (2019). *Asynchronous pipelines for processing huge corpora on medium to low resource infrastructures* (corpus OSCAR). CMLC-7.
- Zhang, T. et al. (2020). *BERTScore: Evaluating Text Generation with BERT*. ICLR (idea de alineación palabra a palabra usada en la variante B).
- Jurafsky, D. & Martin, J. H. *Speech and Language Processing* (3.ª ed. borrador), cap. 6 "Vector Semantics and Embeddings" y cap. 14 "Question Answering". https://web.stanford.edu/~jurafsky/slp3/

**Contenido del dominio**
- Ley 26.427 – Sistema de Pasantías Educativas (Argentina): duración 2–12 meses y renovación de hasta 6 meses (art. 13), hasta 20 h semanales (art. 13), ART (art. 14), asignación estímulo y cobertura de salud (art. 15). Texto: https://www.argentina.gob.ar/normativa/nacional/148599/texto
- Las respuestas sobre el proceso, sedes, beneficios y contacto corresponden a una **empresa ficticia** creada para este trabajo.